# DataGastro
## Primera base analítica del ecosistema gastronómico de la Ciudad de Buenos Aires

**Informe de análisis completo** — construido sobre datos abiertos del Gobierno de la Ciudad de Buenos Aires (GCBA) y relevamientos manuales trazables.

> Datos validados al 10 de junio de 2026. Pipeline de validación: 62 controles aprobados, 0 advertencias, 0 errores.

*Reproducible: Kernel → Restart & Run All reconstruye el informe completo desde los datos del proyecto sin modificar ningún archivo.*

## Cómo leer este informe

La regla central de todo lo que sigue: **las fuentes no se suman cuando miden cosas distintas.**

El ecosistema gastronómico de la Ciudad aparece en varios registros oficiales que describen dimensiones distintas del mismo fenómeno: oferta registrada en guías, habilitaciones formales, espacios públicos, eventos institucionales, programas de gobierno. Cada uno aporta algo valioso, pero mezclarlos produciría conclusiones que los datos no sostienen.

Por eso este informe trabaja cada fuente por separado, explica de dónde viene cada número, y deja en claro qué se puede afirmar con certeza y qué todavía no.

In [ ]:
from pathlib import Path
import json, unicodedata
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly
from matplotlib.collections import PatchCollection
import matplotlib.colors as mcolors
import matplotlib.cm as cmx
import warnings
warnings.filterwarnings('ignore')

# Encontrar la raíz del proyecto (funciona corriendo desde notebooks/ o desde la raíz)
ROOT = Path.cwd()
while not (ROOT / 'data' / 'processed').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / 'data' / 'processed'
ANALYTICS = ROOT / 'data' / 'analytics'
RAW = ROOT / 'data' / 'raw'
print('Raíz del proyecto:', ROOT)

def leer(path):
    p = Path(path)
    return pd.read_csv(p, dtype=str, keep_default_na=False) if p.exists() else pd.DataFrame()

def miles(n):
    return f'{int(n):,}'.replace(',', '.')

def norm(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().upper().strip()

# Cargar tablas analíticas
resumen    = leer(ANALYTICS / 'analytics_resumen_ejecutivo.csv')
hab_anio   = leer(ANALYTICS / 'analytics_habilitaciones_por_anio.csv')
hab_cat    = leer(ANALYTICS / 'analytics_habilitaciones_por_categoria.csv')
est_barrio = leer(ANALYTICS / 'analytics_establecimientos_por_categoria_barrio.csv')
ubic       = leer(PROC / 'dim_ubicacion.csv')
fact_est   = leer(PROC / 'fact_establecimiento.csv')
fact_hab   = leer(PROC / 'fact_habilitacion_gastronomica.csv')
fact_esp   = leer(PROC / 'fact_espacio_feria_mercado.csv')
fact_ev    = leer(PROC / 'fact_evento_gastronomico.csv')
fact_prog  = leer(PROC / 'fact_programa_politica.csv')
anal_prog  = leer(ANALYTICS / 'analytics_programas_catalogo.csv')

def indicador(nombre):
    if resumen.empty or 'indicador' not in resumen.columns: return 0
    fila = resumen[resumen['indicador'] == nombre]
    if fila.empty: return 0
    try: return int(float(fila.iloc[0]['valor']))
    except Exception: return 0

# Coordenadas para los mapas
ubic['lat'] = pd.to_numeric(ubic.get('latitud'), errors='coerce')
ubic['lon'] = pd.to_numeric(ubic.get('longitud'), errors='coerce')
geo = ubic[['id_ubicacion', 'lat', 'lon']].copy()
geo['cg'] = ubic.get('calidad_geo', '')

def puntos(fact, calidad):
    if fact.empty: return pd.DataFrame(columns=['lat','lon'])
    m = fact[['id_ubicacion']].merge(geo, on='id_ubicacion', how='left')
    return m[(m['cg'] == calidad) & m['lat'].notna()]

# Polígonos de barrios para los mapas coropleta
def cargar_barrios():
    p = RAW / 'geo_barrios.geojson'
    if not p.exists(): return []
    feats = json.loads(p.read_text(encoding='utf-8')).get('features', [])
    out = []
    for f in feats:
        g = f['geometry']; coords = g['coordinates']
        parts = coords if g['type'] == 'Polygon' else [r for mp in coords for r in mp]
        rings = [[(pt[0], pt[1]) for pt in ring] for ring in parts]
        out.append({'nombre': f['properties'].get('nombre',''),
                    'area_km2': float(f['properties'].get('area_metro', 0) or 0)/1_000_000,
                    'rings': rings})
    return out
BARRIOS = cargar_barrios()

def contorno(ax):
    pc = [MplPoly(r, closed=True) for b in BARRIOS for r in b['rings']]
    ax.add_collection(PatchCollection(pc, facecolor='#f4f4f4', edgecolor='#bcbcbc', linewidths=0.5))

def encuadrar(ax, x=(-58.54,-58.33), y=(-34.71,-34.53)):
    ax.set_xlim(*x); ax.set_ylim(*y); ax.set_aspect(1/0.82); ax.set_xticks([]); ax.set_yticks([])

def coropleta(ax, valores, titulo, sufijo='', topn=7, gamma=0.6):
    if not valores: return None
    mx = max(valores.values())
    if mx == 0: return None
    ncol = mcolors.PowerNorm(gamma=gamma, vmin=0, vmax=mx)
    cmap = plt.colormaps['YlOrRd']
    patches, colors, etiquetas = [], [], []
    for b in BARRIOS:
        n = norm(b['nombre']); v = valores.get(n, 0)
        for r in b['rings']:
            patches.append(MplPoly(r, closed=True)); colors.append(cmap(ncol(v)))
        if v > 0:
            pts = [pt for r in b['rings'] for pt in r]
            cx = sum(p[0] for p in pts)/len(pts); cy = sum(p[1] for p in pts)/len(pts)
            etiquetas.append((cx, cy, b['nombre'], v))
    ax.add_collection(PatchCollection(patches, facecolor=colors, edgecolor='#888', linewidths=0.5))
    encuadrar(ax); ax.set_title(titulo, fontsize=13)
    top = {n for n,_ in sorted(valores.items(), key=lambda x:-x[1])[:topn]}
    for cx, cy, nombre, v in etiquetas:
        if norm(nombre) in top:
            txt = f'{nombre}\n{int(round(v))}{sufijo}'
            ax.annotate(txt, (cx, cy), ha='center', va='center', fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.78))
    sm = cmx.ScalarMappable(norm=ncol, cmap=cmap); sm.set_array([])
    return sm

print('Datos cargados. Todo listo para el análisis.')

---
## 1. Objetivos del trabajo

La Ciudad de Buenos Aires tiene un sector gastronómico muy activo, pero la información sobre ese sector vive dispersa en registros oficiales que nunca se habían integrado. Hay una guía de oferta gastronómica, un registro de habilitaciones formales, padrones de ferias y mercados, noticias sobre eventos, y documentos sobre programas de gobierno. Cada uno describe algo diferente.

**DataGastro nació para resolver ese problema.** No para producir un número único que sume todo, sino para:

1. **Integrar las fuentes disponibles** en un modelo de datos único, reproducible y trazable.
2. **Separar correctamente los universos** de manera que cada cifra tenga una definición clara y defensible.
3. **Convertir la información en insumos útiles** para conversación pública, planificación territorial y diseño de políticas.
4. **Dejar en claro qué se sabe, qué no se sabe y con qué fuente se sostiene cada afirmación.**

El resultado es una base analítica que permite preparar diagnósticos territoriales rápidos, ordenar evidencia antes de reuniones o recorridas de campo, y construir materiales de presentación interna sin perder las cautelas metodológicas.

---
## 2. Preguntas que guiaron el análisis

Al inicio del proyecto se plantearon estas preguntas concretas. Cada una tiene una respuesta al final del informe (Sección 13).

1. ¿Dónde se concentra la oferta gastronómica registrada en la Ciudad?
2. ¿Qué tipos de gastronomía predominan?
3. ¿Cómo evolucionó la actividad formal (habilitaciones) año a año?
4. ¿Dónde, en términos de calles y barrios, se está aprobando actividad gastronómica formal?
5. ¿Qué espacios públicos de abastecimiento existen y dónde están distribuidos?
6. ¿Qué eventos y programas impulsa la Ciudad en el sector gastronómico?
7. ¿Hay diferencia entre "dónde hay muchos locales" y "dónde están más concentrados"?
8. ¿Qué no se puede responder con estos datos y por qué?

Hay preguntas que el análisis no puede responder todavía — y esa honestidad es parte del valor del trabajo.

---
## 3. Principio metodológico central: las fuentes no se suman

El principal riesgo metodológico al trabajar con estos datos es mezclar universos y comunicar conclusiones que los datos no sostienen. El ejemplo más común: sumar la oferta registrada (F01) con las habilitaciones aprobadas (F02) y decir "hay X establecimientos gastronómicos en la Ciudad". Ese número no existe en los datos porque cada fuente describe algo distinto.

La tabla siguiente resume qué mide cada fuente y cuál es su límite de lectura para una presentación ejecutiva.

In [ ]:
fuentes = pd.DataFrame([
    {'Fuente': 'Oferta gastronómica registrada (F01)',
     'Organismo de origen': 'Ente de Turismo del GCBA',
     'Qué mide': 'Establecimientos publicados en la guía oficial de gastronomía de la Ciudad',
     'Qué NO dice': 'No confirma si cada local sigue abierto hoy'},
    {'Fuente': 'Habilitaciones gastronómicas aprobadas (F02)',
     'Organismo de origen': 'Agencia Gubernamental de Control (AGC)',
     'Qué mide': 'Trámites de habilitación aprobados por rubro gastronómico, 2015–2024',
     'Qué NO dice': 'No son locales activos ni registran cierres'},
    {'Fuente': 'Ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (F03)',
     'Organismo de origen': 'Dirección General de Ferias del GCBA',
     'Qué mide': 'Espacios reales de abastecimiento público en la Ciudad',
     'Qué NO dice': 'No se cuentan puestos ni personas, solo los espacios como tal'},
    {'Fuente': 'Eventos gastronómicos relevados (F04)',
     'Organismo de origen': 'Relevamiento manual con fuente anotada por fila',
     'Qué mide': 'Inventario trazable de eventos del sector, verificados uno a uno',
     'Qué NO dice': 'No es el universo completo de eventos gastronómicos de la Ciudad'},
    {'Fuente': 'Programas y políticas gastronómicas (F05)',
     'Organismo de origen': 'Relevamiento manual con fuente anotada por fila',
     'Qué mide': 'Catálogo institucional de programas vigentes con trazabilidad normativa',
     'Qué NO dice': 'No mide impacto económico, empleo ni presupuesto ejecutado'},
])
pd.set_option('display.max_colwidth', 80)
display(fuentes)

---
## 4. Las fuentes de datos: de dónde viene cada número

Esta sección describe en detalle cada fuente integrada: su origen, cómo se obtuvo, qué contiene, cómo se procesó y cuáles son sus límites de lectura. Este nivel de detalle es importante porque permite defender cada cifra frente a preguntas difíciles.

---

### 4.1 Oferta gastronómica registrada — Fuente 1 (F01)

**Organismo de origen:** Ente de Turismo del Gobierno de la Ciudad de Buenos Aires.

**Cómo se obtuvo:** Descarga directa del portal de datos abiertos de la Ciudad (Buenos Aires Data), formato CSV.

**Qué contiene:** 2.823 registros de establecimientos gastronómicos publicados en la guía oficial de gastronomía de la Ciudad. Incluye nombre, dirección, categoría (restaurante, bar, café, etc.), tipo de cocina, ambientación, horario y datos de contacto.

**Cómo se procesó:** Se corrigió el encoding del archivo (tenía caracteres mal codificados). Se normalizaron las direcciones y se enriqueció la geocodificación. Se asignó categoría gastronómica por tipo de local.

**Límite de lectura:** Este registro es la guía publicada, no un padrón con actualizaciones de vigencia. Un local puede aparecer en la guía y ya no estar operativo, o puede estar operativo y no aparecer porque nunca fue incluido. **No se puede afirmar cuántos locales están activos hoy.**

---

### 4.2 Habilitaciones gastronómicas aprobadas — Fuente 2 (F02)

**Organismo de origen:** Agencia Gubernamental de Control (AGC) del Gobierno de la Ciudad de Buenos Aires.

**Cómo se obtuvo:** Descarga directa del portal Buenos Aires Data, en archivos CSV separados por año (2015–2024). El archivo de 2025 tiene un esquema distinto y mezcla disposiciones de varios años, por lo que se informa aparte.

**Qué contiene:** 44.169 habilitaciones gastronómicas aprobadas. Cada registro es una autorización formal para operar un rubro gastronómico en una dirección determinada. Incluye fecha de habilitación, código y descripción del rubro, superficie del local, ubicación catastral (sección, manzana, parcela) y titulares.

**Cómo se procesó:** Se filtró por categoría gastronómica usando un clasificador por palabra completa con exclusiones trazables (para evitar casos como "talabartería" o "envasados" que contienen palabras de rubros alimenticios pero no son gastronomía de servicio). La categoría "Comercio alimenticio minorista" se separó como no gastronómica y no se cuenta. Se normalizaron las direcciones, que venían con formato invertido ("FERNANDEZ DE LA CRUZ, F., GRAL. AV. 4602") y se geocodificaron con el Sistema de Información Geográfica del GCBA (USIG).

**Geocodificación con USIG:** 42.741 habilitaciones (97% del total) pudieron ubicarse en el mapa mediante el normalizador oficial del GCBA. La tasa de exactitud fue del 98,99% y la consistencia entre la comuna informada por la fuente y la detectada por USIG fue del 100%. Esto convierte a F02 en la única capa que permite ver, calle por calle, dónde se aprueba actividad gastronómica formal.

**Serie temporal comparable:** La serie 2019–2024 es directamente comparable como flujo anual. El período 2015–2018 está consolidado en un único archivo y no es comparable como flujo anual. El archivo de 2025 tampoco es comparable por su esquema distinto. Ambos se muestran aparte en el análisis.

**Límite de lectura:** Una habilitación aprobada no es un local activo. No hay registro de bajas en esta fuente. **No se puede saber cuántos locales están operando hoy ni cuántos cerraron.**

---

### 4.3 Ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial — Fuente 3 (F03)

**Organismo de origen:** Dirección General de Ferias del Ministerio de Ambiente y Espacio Público del GCBA.

**Cómo se obtuvo:** Dos archivos: un CSV con el padrón de puestos y personas, y un archivo GeoJSON con los puntos georreferenciados de las Ferias Itinerantes de Abastecimiento Barrial (FIAB).

**Qué contiene:** La fuente original mezcla recursos con distintos niveles de detalle. Después del procesamiento, se identificaron **259 espacios reales**: 6 mercados municipales, 69 ferias especializadas y 184 puntos de Ferias Itinerantes de Abastecimiento Barrial. Los puestos individuales (4.352 registros con datos personales de feriantes) se usan solo como insumo técnico interno y no se incluyen en los indicadores.

**Las Ferias Itinerantes de Abastecimiento Barrial (FIAB)** son un tipo especial de espacio: ferias de productos frescos y básicos (frutas, verduras, pescado, pan) que rotan entre puntos fijos de cada barrio. Sus 184 puntos están georreferenciados en el archivo GeoJSON oficial.

**Límite de lectura:** Se cuenta el espacio (la feria, el mercado, el punto FIAB), no la cantidad de puestos ni de personas. Dos ferias en el mismo barrio cuentan como dos espacios. **Un espacio no equivale a un establecimiento gastronómico permanente.**

---

### 4.4 Eventos gastronómicos relevados — Fuente 4 (F04)

**Origen:** Relevamiento manual realizado a partir de anuncios oficiales del GCBA (GCBA Noticias, portales de Turismo y Cultura), con la URL de la fuente anotada fila por fila.

**Qué contiene:** 29 eventos cargados, de los cuales 13 están verificados y aptos para métricas. Los 16 restantes están en validación o son cualitativos (sin fecha o ubicación precisa suficiente).

**Tipos de eventos:** festivales gastronómicos, ciclos de mercado, concursos, jornadas de descuentos, participaciones institucionales.

**Límite de lectura:** Este inventario es trazable y verificado, pero **no representa el universo completo de eventos gastronómicos de la Ciudad**. Es un catálogo documentado, no una base estadística.

---

### 4.5 Programas y políticas gastronómicas — Fuente 5 (F05)

**Origen:** Relevamiento manual de instituciones del GCBA (Ministerio de Desarrollo Económico, Ministerio de Cultura, Jefatura de Gabinete), con la normativa de referencia anotada por registro.

**Qué contiene:** 9 programas o políticas cargadas, de las cuales 4 están verificadas y aptas para métricas. Entre ellas: BA Capital Gastronómica, Distrito del Vino, Programa de Bares Notables, y el régimen de permisos de área gastronómica (mesas y sillas en vereda).

**Límite de lectura:** F05 es un catálogo institucional, no una serie temporal de impacto. **No mide resultados económicos, empleo ni presupuesto ejecutado.**

---

### 4.6 Geocodificación con el Sistema de Información Geográfica del GCBA (USIG)

La geocodificación es el proceso de convertir una dirección de texto ("Av. Corrientes 1234") en coordenadas geográficas (latitud y longitud) para ubicarla en el mapa.

El normalizador del Sistema de Información Geográfica del GCBA (USIG) es el servicio oficial de la Ciudad para este propósito. Lo que hace es diferente a un servicio de mapas genérico: normaliza la dirección según la nomenclatura oficial de la Ciudad, valida que el número de calle exista en ese nombre de calle, y devuelve la precisión del resultado (exacta, aproximada, solo lote).

**Por qué fue necesario:** Las direcciones de F02 venían en un formato no estándar (nombre de calle invertido con comas: "FERNANDEZ DE LA CRUZ, F., GRAL. AV. 4602"). Fue necesario desarrollar un paso de normalización previo para convertirlas al formato que acepta USIG.

**Resultado:** 42.741 habilitaciones geocodificadas (97% del universo F02), con tasa de exactitud del 98,99% y consistencia de comuna del 100%. La cache de geocodificación queda guardada en el proyecto para no repetir consultas ya hechas.

---
## 5. Los números principales — separados por fuente

**Fuente:** `data/analytics/analytics_resumen_ejecutivo.csv`, generado por el pipeline de DataGastro a partir de los datos reales cargados.

Estos son los indicadores de cada fuente. No se suman porque cada uno mide algo diferente.

In [ ]:
f01_n     = indicador('establecimientos_oferta_gastronomica_f01')
f02_n     = indicador('habilitaciones_gastronomicas_f02')
f02_geo_n = indicador('habilitaciones_f02_geocodificadas')
f03_n     = indicador('espacios_ferias_mercados_f03')
f04_n     = indicador('eventos_gastronomicos_reales_f04_aptos')
f05_n     = indicador('programas_politicas_reales_f05_aptos')

print('=' * 62)
print('  DATAGASTRO — INDICADORES PRINCIPALES (fuentes separadas)')
print('=' * 62)
print(f'  Oferta gastronómica registrada (F01)         : {miles(f01_n):>8}')
print(f'  Habilitaciones gastronómicas aprobadas (F02) : {miles(f02_n):>8}')
print(f'    de las cuales ubicadas en el mapa (USIG)   : {miles(f02_geo_n):>8}  ({f02_geo_n/f02_n*100:.0f}% del total F02)')
print(f'  Espacios de ferias/mercados/FIAB (F03)       : {miles(f03_n):>8}')
print(f'  Eventos gastronómicos verificados (F04)      : {miles(f04_n):>8}')
print(f'  Programas y políticas vigentes (F05)         : {miles(f05_n):>8}')
print('=' * 62)
print()
print('Recordatorio: F02 son habilitaciones aprobadas, no locales activos.')
print('No se suma F01 + F02 + F03 porque cada uno mide algo diferente.')

df_metricas = pd.DataFrame([
    {'Indicador': 'Oferta registrada (F01)', 'Valor': f01_n},
    {'Indicador': 'Habilitaciones aprobadas (F02)', 'Valor': f02_n},
    {'Indicador': 'Espacios públicos ferias/mercados (F03)', 'Valor': f03_n},
    {'Indicador': 'Eventos verificados (F04)', 'Valor': f04_n},
    {'Indicador': 'Programas vigentes (F05)', 'Valor': f05_n},
]).sort_values('Valor')

fig, ax = plt.subplots(figsize=(10, 4.5))
colores = ['#c07030', '#1f8a4c', '#5c7c3f', '#2b6cb0', '#ba4836']
bars = ax.barh(df_metricas['Indicador'], df_metricas['Valor'], color=colores)
ax.set_title('Indicadores principales — cada barra es un universo distinto, no se suman', fontsize=12)
ax.set_xlabel('Cantidad de registros')
for bar, val in zip(bars, df_metricas['Valor']):
    ax.text(val, bar.get_y() + bar.get_height()/2, f'  {miles(val)}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

**Lectura:** Las habilitaciones (F02) tienen un volumen mucho mayor porque acumulan trámites desde 2015 hasta 2024. La oferta registrada (F01) es una guía estática. Los espacios públicos (F03) son pocos pero geográficamente relevantes. Los eventos y programas (F04, F05) son catálogos documentados, no estadísticas.

---
## 6. El mapa del ecosistema

**Fuentes:** `data/processed/fact_establecimiento.csv` (F01), `data/processed/fact_espacio_feria_mercado.csv` (F03), `data/processed/fact_habilitacion_gastronomica.csv` (F02), coordenadas de `data/processed/dim_ubicacion.csv`. Polígonos de barrios desde `data/raw/geo_barrios.geojson` (fuente oficial del GCBA).

**Cómo se construyó este mapa:** Las coordenadas de F01 y F03 vienen directamente de las fuentes originales. Las coordenadas de F02 se obtuvieron mediante la geocodificación con USIG (ver Sección 4.6). Solo se muestran puntos con calidad geográfica validada.

El panel izquierdo muestra la oferta registrada (azul) y los espacios públicos (verde). El panel derecho suma las habilitaciones geocodificadas (rojo) — la capa más densa porque acumula 10 años de autorizaciones formales.

In [ ]:
f01_pts = puntos(fact_est, 'fuente_oficial')
f03_pts = puntos(fact_esp, 'fuente_oficial')
f02_pts = puntos(fact_hab, 'usig_exacta')

fig, (axa, axb) = plt.subplots(1, 2, figsize=(16, 8))
for ax in (axa, axb):
    contorno(ax); encuadrar(ax)

axa.scatter(f01_pts['lon'], f01_pts['lat'], s=7, c='#2b6cb0', alpha=0.55, linewidths=0,
            label=f'Oferta registrada F01 ({miles(len(f01_pts))})')
axa.scatter(f03_pts['lon'], f03_pts['lat'], s=34, c='#1f8a4c', alpha=0.95, marker='^', linewidths=0,
            label=f'Ferias/mercados/FIAB F03 ({miles(len(f03_pts))})')
axa.set_title('Oferta registrada y espacios públicos (F01 + F03)', fontsize=13)
axa.legend(loc='lower left', fontsize=9, framealpha=0.9)

axb.scatter(f02_pts['lon'], f02_pts['lat'], s=4, c='#ba4836', alpha=0.16, linewidths=0,
            label=f'Habilitaciones USIG F02 ({miles(len(f02_pts))})')
axb.scatter(f01_pts['lon'], f01_pts['lat'], s=5, c='#2b6cb0', alpha=0.45, linewidths=0,
            label=f'Oferta registrada F01 ({miles(len(f01_pts))})')
axb.set_title('Con habilitaciones geocodificadas (F02 via USIG)', fontsize=13)
axb.legend(loc='lower left', fontsize=9, framealpha=0.9)

fig.suptitle('El ecosistema gastronómico sobre el mapa de la Ciudad de Buenos Aires', fontsize=15, y=0.97)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

**Lectura.** La actividad gastronómica se concentra con fuerza en el corredor norte (Palermo, Recoleta, Belgrano) y el centro histórico (San Nicolás, Monserrat, San Telmo), y se afina progresivamente hacia el sur. La capa de habilitaciones geocodificadas (panel derecho) es la única que permite ver, calle por calle, dónde se aprueba actividad gastronómica formal. **Son habilitaciones aprobadas, no locales activos.**

---
## 7. ¿Dónde se concentra la oferta? Volumen absoluto vs. densidad

**Fuentes:** `data/processed/fact_establecimiento.csv` (F01), coordenadas de `data/processed/dim_ubicacion.csv`, polígonos de barrios desde `data/raw/geo_barrios.geojson` (que incluye el campo de superficie en metros cuadrados para calcular densidad).

Hay un matiz que cambia la conclusión sobre dónde está el núcleo gastronómico de la Ciudad. En números absolutos, Palermo manda. Pero Palermo es enorme: tiene mucho espacio para absorber esa oferta. Si se mira la **densidad** (cuántos registros hay por kilómetro cuadrado), el verdadero núcleo es el microcentro y el casco histórico.

In [ ]:
m = fact_est[['id_ubicacion']].merge(ubic[['id_ubicacion','barrio']], on='id_ubicacion', how='left')
m = m[~m['barrio'].isin(['No determinado',''])]
conteo = m['barrio'].map(norm).value_counts().to_dict()
area   = {norm(b['nombre']): b['area_km2'] for b in BARRIOS if b['area_km2'] > 0}
densidad = {n: conteo[n]/area[n] for n in conteo if area.get(n)}

fig, (axa, axb) = plt.subplots(1, 2, figsize=(16, 8))
sm1 = coropleta(axa, conteo,   'Oferta registrada — cantidad absoluta por barrio')
sm2 = coropleta(axb, densidad, 'Oferta registrada — densidad (registros por km²)', sufijo='/km²')
if sm1: fig.colorbar(sm1, ax=axa, fraction=0.04, pad=0.02).set_label('Registros F01 (absoluto)', fontsize=9)
if sm2: fig.colorbar(sm2, ax=axb, fraction=0.04, pad=0.02).set_label('Registros F01 / km²', fontsize=9)
fig.suptitle('Oferta gastronómica registrada (F01) — volumen vs. densidad por barrio', fontsize=15, y=0.97)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

top_abs = sorted(conteo.items(), key=lambda x:-x[1])[:6]
top_den = sorted(densidad.items(), key=lambda x:-x[1])[:6]
print('Top 6 en volumen absoluto (F01):')
for n, v in top_abs: print(f'  {n.title()}: {v} registros')
print()
print('Top 6 en densidad (F01, registros por km²):')
for n, v in top_den: print(f'  {n.title()}: {v:.0f} registros/km²')

**Lectura.** Palermo lidera en volumen absoluto, pero por su tamaño la oferta queda relativamente diluida. Por densidad, **San Nicolás multiplica por seis o siete la concentración de Palermo**: el corazón gastronómico por intensidad es el microcentro y el casco histórico (San Nicolás, Monserrat, San Telmo).

¿Cuándo importa cada lectura?
- El **volumen absoluto** es relevante para entender dónde hay más opciones en total.
- La **densidad** es relevante para entender dónde el sector está más comprimido en el espacio — lo que tiene implicancias para política de uso del suelo, permisos, tránsito y servicios.

---
## 8. ¿Cuánto se habilita por año? La evolución formal del sector

**Fuente:** `data/analytics/analytics_habilitaciones_por_anio.csv`, generado desde los registros de la Agencia Gubernamental de Control (AGC) disponibles en Buenos Aires Data.

Una habilitación aprobada es la autorización formal para operar un rubro en un domicilio. Marca dónde el sector formal está invirtiendo. El gráfico muestra solo los años comparables entre sí como flujo anual (2019–2024). Los períodos con estructura distinta se muestran aparte.

In [ ]:
serie = hab_anio.copy()
if 'comparable_como_flujo_anual' in serie.columns:
    comp   = serie[serie['comparable_como_flujo_anual'].astype(str) == 'si'].copy()
    nocomp = serie[serie['comparable_como_flujo_anual'].astype(str) == 'no'].copy()
else:
    comp, nocomp = serie, serie.iloc[0:0]

comp['n'] = pd.to_numeric(comp['cantidad_habilitaciones'], errors='coerce').fillna(0).astype(int)
comp = comp.sort_values('anio_fuente')

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(comp['anio_fuente'].astype(str), comp['n'], color='#5c7c3f', width=0.6)
ax.set_title('Habilitaciones gastronómicas aprobadas por año — serie comparable 2019–2024', fontsize=13)
ax.set_ylabel('Habilitaciones aprobadas')
ax.set_xlabel('Año')
media = comp['n'].mean()
ax.axhline(media, color='#ba4836', linestyle='--', linewidth=1.5, label=f'Promedio del período: {miles(int(media))}')
ax.legend(fontsize=10)
for bar, v in zip(bars, comp['n']):
    ax.annotate(miles(v), (bar.get_x() + bar.get_width()/2, v),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

if not nocomp.empty:
    print('Períodos que se informan por separado (esquema distinto, no comparables como flujo anual):')
    for _, r in nocomp.iterrows():
        anio = r['anio_fuente']
        cant = miles(r['cantidad_habilitaciones'])
        nota = r.get('nota_serie', '') if 'nota_serie' in r.index else ''
        print(f'  - Período {anio}: {cant} habilitaciones. {nota}')

**Lectura.** La serie comparable (2019–2024) permite ver la dinámica anual del sector formal: cuántas autorizaciones se aprobaron cada año, con el impacto visible de la pandemia de COVID-19 en 2020. La línea punteada muestra el promedio del período para facilitar la comparación entre años.

**Por qué 2015–2018 y 2025 se muestran aparte:** El archivo del período 2015–2018 está consolidado en un único CSV sin desagregación anual. El archivo de 2025 tiene un esquema de columnas distinto y mezcla disposiciones de varios años, por lo que no es comparable como flujo anual. **No se pueden mezclar en el mismo gráfico sin falsear la comparación.**

---
## 9. ¿Qué tipo de gastronomía se habilita?

**Fuente:** `data/analytics/analytics_habilitaciones_por_categoria.csv`, generado desde los registros de la Agencia Gubernamental de Control (AGC). La categoría gastronómica es inferida desde la descripción del rubro usando un clasificador por palabra completa con exclusiones trazables.

**Nota metodológica:** El clasificador fue rediseñado durante el proyecto para evitar falsos positivos. La versión anterior inflaba el total a 87.934 porque asignaba categorías gastronómicas a rubros como "talabartería" (que contiene la raíz de una palabra alimenticia) o "envasados" (que se confundía con bares). El clasificador actual usa coincidencia por palabra completa y llega a 44.169 habilitaciones genuinamente gastronómicas.

In [ ]:
cat = hab_cat.copy()
cat['n'] = pd.to_numeric(cat['cantidad_habilitaciones'], errors='coerce').fillna(0).astype(int)
cat = cat[cat['n'] > 0].sort_values('n', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(cat['categoria_gastronomica_inferida'], cat['n'], color='#2f6f9f')
ax.set_title('Habilitaciones gastronómicas aprobadas por categoría (F02 — Agencia Gubernamental de Control)', fontsize=12)
ax.set_xlabel('Cantidad de habilitaciones aprobadas (2015–2024)')
for bar, v in zip(bars, cat['n']):
    ax.text(v, bar.get_y() + bar.get_height()/2, f'  {miles(v)}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

**Lectura.** La categoría más habilitada revela cuál es el rubro gastronómico con más actividad formal en la Ciudad a lo largo del período. La categoría "Comercio alimenticio minorista" (venta de alimentos sin servicio gastronómico de mesa) se separó del conteo gastronómico principal porque no representa el mismo tipo de actividad.

---
## 10. Espacios públicos: ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (FIAB)

**Fuente:** `data/processed/fact_espacio_feria_mercado.csv`, consolidado desde el CSV de la Dirección General de Ferias del GCBA y el archivo GeoJSON oficial de puntos de Ferias Itinerantes de Abastecimiento Barrial (FIAB).

Este es el componente público y territorial del ecosistema: los espacios que la Ciudad gestiona directamente para la comercialización de alimentos. Se cuentan espacios reales (la feria como unidad), no los puestos individuales ni las personas.

In [ ]:
if not fact_esp.empty and 'tipo_espacio' in fact_esp.columns:
    tipo = fact_esp['tipo_espacio'].value_counts().reset_index()
    tipo.columns = ['Tipo de espacio', 'Cantidad']

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(tipo['Tipo de espacio'][::-1], tipo['Cantidad'][::-1], color='#1f8a4c')
    ax.set_title('Espacios de abastecimiento público por tipo (F03 — Dirección General de Ferias)', fontsize=12)
    ax.set_xlabel('Cantidad de espacios reales')
    for bar, v in zip(bars, tipo['Cantidad'][::-1]):
        ax.text(v, bar.get_y() + bar.get_height()/2, f'  {miles(v)}', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    print('Detalle de espacios F03:')
    for _, row in tipo.iterrows():
        print(f'  - {row["Tipo de espacio"]}: {miles(row["Cantidad"])} espacios reales')
    print()
    print('Las Ferias Itinerantes de Abastecimiento Barrial (FIAB) son puntos fijos')
    print('donde ferias de productos básicos (frutas, verduras, pescado, pan) rotan')
    print('por los barrios según un cronograma oficial.')
else:
    print('Archivo de espacios F03 no encontrado.')

**Lectura.** Los 259 espacios reales se distribuyen en tres tipos: mercados municipales (permanentes, con infraestructura propia), ferias especializadas (según rubro: artesanías, libros, ropa, con algunos mixtos alimentarios) y los 184 puntos de Ferias Itinerantes de Abastecimiento Barrial (FIAB), que son la red de abastecimiento de proximidad de la Ciudad. El GeoJSON oficial de FIAB permite ver exactamente en qué esquina opera cada punto.

---
## 11. Eventos y programas que impulsa la Ciudad

**Fuentes:** `data/processed/fact_evento_gastronomico.csv` (F04) y `data/processed/fact_programa_politica.csv` (F05). Ambas son bases de relevamiento manual con la URL de origen anotada por cada fila.

Estos datos muestran la dimensión institucional del ecosistema: qué acciones impulsa el Gobierno de la Ciudad para el sector gastronómico.

In [ ]:
# Eventos verificados (aptos para métricas)
ev_aptos = fact_ev[fact_ev.get('apto_dashboard', pd.Series(dtype=str)).astype(str) == 'si'] \
           if not fact_ev.empty else pd.DataFrame()

print(f'Eventos relevados en total: {len(fact_ev)}')
print(f'Eventos verificados (aptos para métricas): {len(ev_aptos)}')
print()

if not ev_aptos.empty and 'tipo_evento' in ev_aptos.columns:
    print('Eventos verificados por tipo:')
    for tipo, cnt in ev_aptos['tipo_evento'].value_counts().items():
        print(f'  - {tipo}: {cnt}')

if not ev_aptos.empty and 'nombre_evento' in ev_aptos.columns:
    cols_ev = [c for c in ['nombre_evento', 'tipo_evento', 'barrio', 'anio'] if c in ev_aptos.columns]
    print()
    print('Lista de eventos verificados:')
    display(ev_aptos[cols_ev].reset_index(drop=True))

print()
print('Nota: con 13 eventos verificados esto muestra que existe un calendario sostenido,')
print('pero no es una base estadística — es un inventario documentado.')

In [ ]:
# Programas y políticas vigentes
if not anal_prog.empty:
    cols_prog = [c for c in ['nombre_programa', 'tipo_programa', 'estado', 'organismo_responsable'] if c in anal_prog.columns]
    prog_aptos = anal_prog[anal_prog.get('apto_dashboard', pd.Series(dtype=str)).astype(str) == 'si'] \
                 if 'apto_dashboard' in anal_prog.columns else anal_prog
    print(f'Programas y políticas en el catálogo: {len(anal_prog)}')
    print(f'Programas verificados y vigentes: {len(prog_aptos)}')
    print()
    print('Programas vigentes:')
    display(prog_aptos[cols_prog].reset_index(drop=True))
else:
    print('Catálogo de programas no encontrado.')

**Lectura.** Los eventos y programas muestran que la Ciudad tiene una agenda gastronómica activa: festivales, ciclos de mercado, concursos, regímenes de incentivos territoriales (Distrito del Vino), programas de patrimonio (Bares Notables), y un sistema de permisos de uso del espacio público para mesas y sillas en vereda. **Este es un inventario documentado y trazable, no un universo estadísticamente completo.**

---
## 12. Qué falta y cómo se podría complementar

Una base analítica honesta no solo muestra lo que sabe: también deja en claro qué no puede responder todavía y cómo se podría avanzar. Esta sección describe las brechas principales y las fuentes o acciones que las cubrirían.

### 12.1 Brecha central: no hay padrón de locales activos con bajas

**El problema:** Hoy no existe ninguna fuente que permita saber cuántos establecimientos gastronómicos están efectivamente operativos en la Ciudad. La oferta registrada (F01) es una guía estática. Las habilitaciones (F02) no registran bajas.

**Cómo se podría cubrir:** Los **permisos de área gastronómica** (mesas y sillas en la vía pública) son un trámite anual que la Ciudad gestiona. Un local que renueva su permiso de mesas probablemente sigue abierto. Esta fuente (denominada F06 en la hoja de ruta del proyecto) sería la señal de actividad vigente más cercana a un padrón vivo.

### 12.2 Brecha en dinámica: las habilitaciones no capturan cierres

**El problema:** F02 registra autorizaciones aprobadas, pero no bajas de habilitación. Tampoco registra aperturas netas (cuántos locales realmente abrieron después de obtener la habilitación). Por eso no se puede calcular evolución neta del stock gastronómico.

**Cómo se podría cubrir:** Solicitar a la Agencia Gubernamental de Control (AGC) el registro de bajas de habilitación como fuente complementaria. Cruzar habilitaciones con el Registro Fiscal de la Administración Gubernamental de Ingresos Públicos (AGIP) también podría aportar información sobre actividad declarada.

### 12.3 Brecha de impacto: sin datos de empleo, ventas ni facturación

**El problema:** DataGastro no mide el impacto económico del sector. No hay datos de cuánto empleo genera, cuánto factura ni cuánto aporta al producto bruto de la Ciudad.

**Cómo se podría cubrir:** Fuentes del Ministerio de Trabajo de Nación (empleo registrado por sector CIIU), datos de AFIP (monotributo y responsables inscriptos en actividades gastronómicas), o encuestas sectoriales específicas (como las de la Asociación de Hoteles, Restaurantes, Confiterías y Cafés, AHRCC).

### 12.4 Brecha en eventos: sin estructura oficial

**El problema:** La agenda de eventos gastronómicos no existe como dataset estructurado. F04 es un relevamiento manual que documenta lo que se puede encontrar en noticias y comunicados oficiales, pero no captura el universo completo.

**Cómo se podría cubrir:** Colaboración con las áreas de Turismo y Cultura del GCBA para sistematizar la agenda de eventos en un dataset actualizable. El Gobierno ya produce esa agenda en formato web — el paso siguiente sería estructurarla.

### 12.5 Brecha territorial: sin denominador de demanda

**El problema:** No se puede saber si un barrio está saturado o subatendido gastronómicamente porque no hay un denominador de demanda (cuántas personas viven, trabajan o visitan cada zona).

**Cómo se podría cubrir:** Datos de flujo peatonal de la Ciudad (que el GCBA mide en algunos corredores), datos de turismo por destino, o una estimación basada en la población y el empleo informal por zona. La densidad de habilitaciones por habitante barrial ya sería un primer paso útil.

### 12.6 Brecha en F02 2025: esquema distinto, no comparable

**El problema:** El archivo de habilitaciones de 2025 publicado por la Agencia Gubernamental de Control tiene una estructura de columnas diferente a los años anteriores y mezcla disposiciones de varios períodos. No se puede incorporar a la serie comparable sin distorsionarla.

**Cómo se podría cubrir:** Esperar una publicación normalizada por parte de la fuente, o solicitar clarificación directamente al área de datos de la AGC para entender el esquema nuevo y ajustar el procesamiento.

---

| Brecha | Por qué importa | Acción posible |
|---|---|---|
| Sin padrón de locales activos | No se sabe cuántos establecimientos siguen abiertos | Incorporar permisos de área gastronómica (F06) |
| F02 no registra cierres | Las habilitaciones no representan aperturas netas | Solicitar registro de bajas a la AGC |
| Sin datos de empleo o facturación | No se puede medir impacto económico | Ministerio de Trabajo, AFIP, encuestas sectoriales |
| Agenda de eventos sin estructura | F04 es inventario parcial | Colaboración con Turismo/Cultura para dataset |
| Sin denominador de demanda | No se puede saber si un barrio está saturado | Flujo peatonal, turismo, empleo por zona |
| F02 2025 no comparable | Distorsiona la serie temporal | Actualización/clarificación de esquema con AGC |

---
## 13. Respuestas a las preguntas iniciales

Al inicio del informe se plantearon ocho preguntas. Acá están las respuestas, con el respaldo de datos de cada una.

---

**Pregunta 1: ¿Dónde se concentra la oferta gastronómica registrada en la Ciudad?**

La oferta registrada (F01) se concentra principalmente en el corredor norte — Palermo, Recoleta, Belgrano — y en el centro histórico — San Nicolás, Monserrat, San Telmo. Palermo tiene el mayor número absoluto de registros. Estos barrios también coinciden con mayor presencia de habilitaciones geocodificadas (F02), lo que sugiere coherencia entre las fuentes.

---

**Pregunta 2: ¿Qué tipos de gastronomía predominan?**

Las categorías con mayor cantidad de habilitaciones aprobadas (F02) son los restaurantes, bares y confiterías, y los locales de comidas rápidas. En la oferta registrada (F01) también predominan restaurantes y bares, con presencia notable de cafeterías y heladerías. La distribución exacta por categoría se ve en el gráfico de la Sección 9.

---

**Pregunta 3: ¿Cómo evolucionó la actividad formal (habilitaciones) año a año?**

La serie comparable 2019–2024 muestra la evolución anual de habilitaciones aprobadas por la Agencia Gubernamental de Control. Se observa el impacto de la pandemia en 2020 (caída significativa) y la recuperación posterior. El promedio del período permite contextualizar los valores de cada año. Los períodos 2015–2018 y 2025 se muestran aparte por tener estructuras no comparables.

---

**Pregunta 4: ¿Dónde, en términos de calles y barrios, se está aprobando actividad gastronómica formal?**

El 97% de las habilitaciones (F02) fueron ubicadas en el mapa mediante el Sistema de Información Geográfica del GCBA (USIG), con una tasa de exactitud del 99%. La capa resultante permite ver, calle por calle, dónde se aprueba actividad gastronómica formal. Las comunas con mayor densidad de habilitaciones son las del centro y el corredor norte. La tasa de consistencia entre la comuna informada por la fuente y la detectada por el normalizador fue del 100%.

---

**Pregunta 5: ¿Qué espacios públicos de abastecimiento existen y dónde están distribuidos?**

La Ciudad gestiona 259 espacios reales: 6 mercados municipales, 69 ferias especializadas y 184 puntos de Ferias Itinerantes de Abastecimiento Barrial (FIAB). Las FIAB están georreferenciadas y distribuidas en todos los barrios con un enfoque de proximidad. Los mercados tienen la mayor permanencia e infraestructura propia. La distribución por comuna muestra una presencia razonablemente equilibrada en comparación con la oferta privada, que está mucho más concentrada en el norte.

---

**Pregunta 6: ¿Qué eventos y programas impulsa la Ciudad en el sector gastronómico?**

Se documentaron 13 eventos verificados (festivales, ciclos de mercado, concursos, jornadas de descuentos) y 4 programas vigentes: BA Capital Gastronómica (programa marco de promoción), Distrito del Vino (incentivos fiscales en la Comuna 11), Programa de Bares Notables (protección patrimonial), y el régimen de permisos de área gastronómica (mesas y sillas en vereda). El catálogo es trazable — cada registro tiene su fuente anotada — pero no representa el universo completo.

---

**Pregunta 7: ¿Hay diferencia entre "dónde hay muchos locales" y "dónde están más concentrados"?**

Sí, y es una diferencia significativa. En números absolutos, Palermo lidera la oferta registrada. Pero por densidad (registros por kilómetro cuadrado), el barrio de San Nicolás concentra entre seis y siete veces más oferta gastronómica que Palermo. El microcentro y el casco histórico (San Nicolás, Monserrat, San Telmo) son el núcleo gastronómico real por intensidad. Esta diferencia tiene implicancias concretas para política de uso del suelo, gestión del tránsito, y evaluación de permisos de espacio público.

---

**Pregunta 8: ¿Qué no se puede responder con estos datos y por qué?**

DataGastro no puede responder hoy:

- **Cuántos locales gastronómicos están activos** hoy en la Ciudad. No existe un padrón con bajas actualizado.
- **Cuántos abrieron o cerraron en términos netos.** Las habilitaciones (F02) no registran bajas.
- **El impacto económico del sector:** empleo, ventas, facturación o contribución al producto bruto de la Ciudad.
- **Si un barrio está saturado o subatendido** gastronómicamente. Para eso hace falta un denominador de demanda que todavía no está integrado.
- **El impacto causal de eventos o programas.** No hay métricas de resultado publicadas para la mayoría de los programas relevados.

Esta honestidad sobre los límites es lo que hace confiable la base: cada afirmación está respaldada, y las que no tienen respaldo no se hacen.

---
## 14. Situación actual y hacia dónde seguir

### Estado del proyecto al cierre de este informe (junio 2026)

DataGastro tiene hoy una base sólida y validada:

- **Pipeline de datos completo:** descarga de fuentes → limpieza y normalización → modelo normalizado → validaciones de calidad → tablas analíticas. El proceso se puede repetir cuando se actualicen los datos.
- **62 controles de validación** aprobados sin errores ni advertencias (`validate_model.py --strict-real`).
- **22 tests automáticos** que verifican normalización de categorías, geocodificación y consistencia del modelo.
- **Geocodificación de F02 completa:** 42.741 habilitaciones ubicadas en el mapa con el normalizador oficial del GCBA.
- **Dashboard Streamlit** disponible para exploración interactiva de los mismos datos de este informe.

### Próximos pasos en orden de prioridad

**1. Incorporar permisos de área gastronómica (Fuente 6, F06)**

Los permisos de mesas y sillas en la vía pública son un trámite anual que la Ciudad gestiona. Un local que renueva su permiso está, casi por definición, activo. Incorporar esta fuente como un universo separado cubriría la principal brecha actual: la ausencia de un padrón de locales con actividad vigente.

**2. Exportar el informe a PDF/HTML para circulación interna**

Esta notebook se puede convertir en un documento HTML o PDF con el siguiente comando desde la raíz del proyecto:

```bash
python -m nbconvert --to html notebooks/07_informe_completo.ipynb
```

**3. Módulo exploratorio de análisis de redes**

Como paso posterior, se puede desarrollar una notebook exploratoria que aplique herramientas de análisis de grafos (centralidad, comunidades, corredores) sobre los puntos geocodificados de F02. El objetivo sería detectar polos gastronómicos y estructuras espaciales que no son visibles en los mapas de puntos. **Este módulo es exploratorio — sus resultados no serían indicadores oficiales.**

### Para explorar los datos en forma interactiva

El proyecto incluye un dashboard en Streamlit para explorar los mismos datos de este informe: mapa con capas que se prenden y apagan, filtros por categoría y comuna, y la coropleta de habilitaciones. Esta notebook es la pieza para **contar la historia**; el dashboard es para **explorar**.

```bash
python -m streamlit run dashboard/app.py
```

---
## 15. Trazabilidad y calidad de los datos

**Fuente:** `data/analytics/analytics_resumen_ejecutivo.csv` — tabla de resumen generada por el pipeline con metadatos de calidad.

Cada tabla del proyecto conserva campos de control de calidad: estado de los datos, fuentes utilizadas, fecha de consulta y si el resultado es apto para uso en métricas. La tabla siguiente muestra ese estado para los bloques de análisis de este informe.

In [ ]:
trazabilidad = []
for nombre_bloque, df in [
    ('Resumen ejecutivo', resumen),
    ('Oferta por barrio/categoría (F01)', est_barrio),
    ('Habilitaciones por año (F02)', hab_anio),
    ('Habilitaciones por categoría (F02)', hab_cat),
    ('Espacios ferias/mercados (F03)', fact_esp),
    ('Eventos gastronómicos (F04)', fact_ev),
    ('Programas y políticas (F05)', anal_prog),
]:
    fila = {
        'Bloque de análisis': nombre_bloque,
        'Estado de datos': df['estado_datos'].iloc[0] if not df.empty and 'estado_datos' in df.columns else 'No disponible',
        'Fuentes utilizadas': df['fuentes_utilizadas'].iloc[0] if not df.empty and 'fuentes_utilizadas' in df.columns else 'No disponible',
        'Fecha máx. consulta': df['fecha_consulta_max'].iloc[0] if not df.empty and 'fecha_consulta_max' in df.columns else 'No disponible',
        'Apto para métricas': df['apto_dashboard'].iloc[0] if not df.empty and 'apto_dashboard' in df.columns else 'No disponible',
    }
    trazabilidad.append(fila)

pd.set_option('display.max_colwidth', 60)
display(pd.DataFrame(trazabilidad))

---

## Glosario de siglas y abreviaturas

| Sigla | Significado completo |
|---|---|
| GCBA | Gobierno de la Ciudad de Buenos Aires |
| AGC | Agencia Gubernamental de Control (del GCBA) |
| USIG | Sistema de Información Geográfica (del GCBA) — normalizador oficial de direcciones |
| FIAB | Ferias Itinerantes de Abastecimiento Barrial |
| F01 | Fuente 1: Oferta gastronómica registrada (Ente de Turismo del GCBA) |
| F02 | Fuente 2: Habilitaciones gastronómicas aprobadas (Agencia Gubernamental de Control) |
| F03 | Fuente 3: Ferias, mercados y Ferias Itinerantes de Abastecimiento Barrial (Dirección General de Ferias) |
| F04 | Fuente 4: Eventos gastronómicos relevados (inventario manual trazable) |
| F05 | Fuente 5: Programas y políticas gastronómicas (catálogo manual trazable) |
| F06 | Fuente 6: Permisos de área gastronómica — pendiente de incorporación |
| CSV | Archivo de datos en formato de texto con valores separados por comas |
| GeoJSON | Formato estándar de datos geoespaciales |
| km² | Kilómetros cuadrados |

---

*DataGastro — datos abiertos del GCBA y relevamientos trazables. Cada número informa su fuente, su fecha y sus límites.*

Para exportar este informe a HTML:
```bash
python -m nbconvert --to html notebooks/07_informe_completo.ipynb
```